In [ ]:
import os
import numpy as np
import pandas as pd
import cv2
import math
import matplotlib.pyplot as plt

meta_data = pd.read_csv('/kaggle/input/cbis-ddsm-breast-cancer-image-dataset/csv/meta.csv')
#meta_data.head()
di_data=pd.read_csv('/kaggle/input/cbis-ddsm-breast-cancer-image-dataset/csv/dicom_info.csv')
#di_data.head()
image_dir='/kaggle/input/cbis-ddsm-breast-cancer-image-dataset/jpeg'
di_data['SeriesDescription'].unique()

#Filtering dataset based on Series Description to extract specific types of images
cropped_images=di_data[di_data.SeriesDescription=='cropped images'].image_path
full_mammo_images=di_data[di_data.SeriesDescription=='full mammogram images'].image_path
ROI_mask_images=di_data[di_data.SeriesDescription=='ROI mask images'].image_path

#Updating the paths of the di_data images 
cropped_images=cropped_images.replace('CBIS-DDSM/jpeg',image_dir, regex=True)
full_mammo_images=full_mammo_images.replace('CBIS-DDSM/jpeg',image_dir,regex=True)
ROI_mask_images=ROI_mask_images.replace('CBIS-DDSM/jpeg',image_dir,regex=True)

#Creating dictionaries where the keys are derived from the paths of the di_data images, where the path is split be /.
full_mammo_images_dict=dict()
cropped_images_dict=dict()
ROI_mask_images_dict=dict()

for data1 in full_mammo_images:
    key=data1.split("/")[5]
    full_mammo_images_dict[key]=data1 
for data1 in cropped_images:
    key=data1.split("/")[5]
    cropped_images_dict[key]=data1   
for data1 in ROI_mask_images:
    key=data1.split("/")[5]
    ROI_mask_images_dict[key]=data1 
     
cal_test_data=pd.read_csv('/kaggle/input/cbis-ddsm-breast-cancer-image-dataset/csv/calc_case_description_test_set.csv')
#cal_test_data.head()
mass_test_data=pd.read_csv('/kaggle/input/cbis-ddsm-breast-cancer-image-dataset/csv/mass_case_description_test_set.csv')
#mass_test_data.head()

cal_train_data=pd.read_csv('/kaggle/input/cbis-ddsm-breast-cancer-image-dataset/csv/calc_case_description_train_set.csv')
#cal_train_data.head()

mass_train_data=pd.read_csv('/kaggle/input/cbis-ddsm-breast-cancer-image-dataset/csv/mass_case_description_train_set.csv')
#mass_train_data.head()

#Update specific columns in a dataset based on mappings stored in previously defined dictionaries


def fix_image_path(data):
    for i, img in enumerate(data.values):
        img_name=img[11].split("/")[2]
        data.iloc[i,11]=full_mammo_images_dict[img_name]
            
        img_name=img[12].split("/")[2]
        data.iloc[i,12]=cropped_images_dict[img_name]
        
        img_name=img[13].split("/")[2]
        data.iloc[i,13]=ROI_mask_images_dict[img_name]

#Applying the function defined above to update the images in the mass datasets
fix_image_path(mass_test_data)
fix_image_path(mass_train_data)

mass_test=mass_test_data.rename(columns={'left or right breast':'left_or_right_breast',
'image view':'image_view','abnormality id':'abnormality_id','mass shape':'mass_shape',
'mass margins':'mass_margins','image file path':'image_file_path',
'cropped image file path':'cropped_image_file_path',
'ROI mask file path':'ROI_mask_file_path'})

mass_train=mass_train_data.rename(columns={'left or right breast':'left_or_right_breast',
'image view':'image_view','abnormality id':'abnormality_id','mass shape':'mass_shape',
'mass margins':'mass_margins','image file path':'image_file_path',
'cropped image file path':'cropped_image_file_path',
'ROI mask file path':'ROI_mask_file_path'})

# Keep mass_test as your test set
test_df = mass_test

print("Test size:", len(test_df))

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Conv2DTranspose, concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Dropout, Conv2DTranspose, concatenate

def conv_block(input, num_filters):
    conv = Conv2D(num_filters, (3, 3), activation="relu", padding="same", kernel_initializer='he_normal')(input)
    conv = Conv2D(num_filters, (3, 3), activation="relu", padding="same", kernel_initializer='he_normal')(conv)
    return conv

def encoder_block(input, num_filters):
    conv = conv_block(input, num_filters)
    pool = MaxPooling2D((2, 2))(conv)
    return conv, pool

def decoder_block(input, skip_features, num_filters):
    uconv = Conv2DTranspose(num_filters, (2, 2), strides=2, padding="same")(input)
    con = concatenate([uconv, skip_features])
    conv = conv_block(con, num_filters)
    return conv

def unet_model(input_shape):
    input_layer = Input(input_shape)
    
    s1, p1 = encoder_block(input_layer, 64)
    s2, p2 = encoder_block(p1, 128)
    s3, p3 = encoder_block(p2, 256)
    s4, p4 = encoder_block(p3, 512)

    b1 = conv_block(p4, 1024)

    d1 = decoder_block(b1, s4, 512)
    d2 = decoder_block(d1, s3, 256)
    d3 = decoder_block(d2, s2, 128)
    d4 = decoder_block(d3, s1, 64)
    
    output_layer = Conv2D(1, 1, padding="same", activation="sigmoid")(d4)

    model = Model(input_layer, output_layer, name="U-Net")
    return model

model = unet_model(input_shape=(256, 256, 1))
model.load_weights("/kaggle/input/tfrecordmodel/tensorflow2/default/1/tfrecordcrops-tpu-unetf.weights.h5")
print("Weights loaded successfully")

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt

def load_and_preprocess_image(img_path):
    img = tf.io.read_file(img_path)
    img = tf.image.decode_png(img, channels=1)           # Grayscale
    img = tf.image.resize(img, [256, 256])               # Resize to match model input
    img = tf.image.convert_image_dtype(img, tf.float32)  # Normalize to [0, 1]
    img = tf.expand_dims(img, axis=0)                    # Add batch dimension
    return img

#image_path = "/kaggle/input/display-cbis-ddsm/image.png"
#input_image = load_and_preprocess_image(image_path)

# Select a sample from the test_df (e.g., first image)
sample = test_df.iloc[110]
image_path = sample["image_file_path"]

print("Selected image path:", image_path)

# Now use this image path in your preprocessing function
input_image = load_and_preprocess_image(image_path)

In [ ]:
import os
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

def predict_full_image_mask(model, image_path, n=4, out_size=256):
    """
    Predicts a full-size mask for an image using patch-wise inference.

    Args:
        model: Trained Keras model.
        image_path (str): Path to the input image.
        n (int): Grid divisions (n x n crops).
        out_size (int): Size to resize each crop for the model input.

    Returns:
        img_np (np.ndarray): Original grayscale image.
        final_mask (np.ndarray): Predicted binary mask with same shape as image.
    """
    # Load image
    img = Image.open(image_path).convert("L")
    img_np = np.array(img)
    original_h, original_w = img_np.shape

    # Pad to make divisible by n
    pad_h = (n - original_h % n) % n
    pad_w = (n - original_w % n) % n
    padded_img = np.pad(img_np, ((0, pad_h), (0, pad_w)), mode='constant', constant_values=0)

    padded_h, padded_w = padded_img.shape
    step_h, step_w = padded_h // n, padded_w // n

    # Create full-size padded mask
    full_mask = np.zeros((padded_h, padded_w), dtype=np.float32)

    # Predict on each crop
    for i in range(n):
        for j in range(n):
            y_start = i * step_h
            x_start = j * step_w

            crop = padded_img[y_start:y_start + step_h, x_start:x_start + step_w]
            resized_crop = cv2.resize(crop, (out_size, out_size))
            input_crop = resized_crop[np.newaxis, ..., np.newaxis] / 255.0  # Normalize

            pred = model.predict(input_crop, verbose=0)[0, ..., 0]
            pred_resized = cv2.resize(pred, (step_w, step_h))  # Resize back

            full_mask[y_start:y_start + step_h, x_start:x_start + step_w] = pred_resized

    # Crop back to original image size
    final_mask = full_mask[:original_h, :original_w]

    return img_np, final_mask

from PIL import Image

def display_and_save_mask(predicted_mask, n, save_dir="/kaggle/working"):
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, f"predicted_mask_n{n}.png")

    # Save using PIL without resizing or cropping
    Image.fromarray((predicted_mask * 255).astype(np.uint8)).save(save_path)

    # Optional: still show with matplotlib
    plt.figure(figsize=(6, 6))
    plt.imshow(predicted_mask, cmap="gray", vmin=0, vmax=1)
    plt.axis("off")
    plt.show()

    print(f"Saved predicted mask to: {save_path}")

import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import Image as IPyImage, display

def combine_masks_by_pixel_count_threshold(base_dir="/kaggle/working", n_range=range(2, 11), threshold=6):
    # Collect all mask paths
    mask_files = [f"predicted_mask_n{n}.png" for n in n_range]
    mask_paths = [os.path.join(base_dir, f) for f in mask_files if os.path.exists(os.path.join(base_dir, f))]

    if not mask_paths:
        print("No predicted mask files found.")
        return

   # print(f"Found {len(mask_paths)} mask files.")

    # Load and binarize masks
    mask_stack = []
    for path in mask_paths:
        mask_img = Image.open(path).convert("L")
        mask_array = np.array(mask_img) / 255.0
        binary_mask = (mask_array > 0.5).astype(np.uint8)
        mask_stack.append(binary_mask)

    mask_stack = np.stack(mask_stack, axis=0)  # Shape: (num_masks, H, W)

    # Count how many times each pixel is white
    white_pixel_count = np.sum(mask_stack, axis=0)

 #   print("White pixel count per pixel:")
  #  print("Min:", white_pixel_count.min(), "Max:", white_pixel_count.max())
  #  print("Applying threshold:", threshold)

    # Create final mask: white if pixel is white in ≥ threshold masks
    final_mask = (white_pixel_count >= threshold).astype(np.uint8)

    # Save safely preserving original shape
    combined_path = os.path.join(base_dir, f"combined_mask_thresholded{threshold}.png")
    Image.fromarray((final_mask * 255).astype(np.uint8)).save(combined_path)

    print(f"Combined mask saved to: {combined_path}")

        
    # Display the result
    plt.figure(figsize=(6, 6))
    plt.imshow(final_mask, cmap="gray")
    #plt.title("Combined Mask (AND across n=2 to 10)")
    plt.axis("off")

import numpy as np

def calculate_iou(mask1, mask2):
    """
    Calculates the Intersection over Union (IoU) between two binary masks.

    Args:
        mask1 (np.ndarray): First binary mask (0 or 255).
        mask2 (np.ndarray): Second binary mask (0 or 255).

    Returns:
        float: IoU score between 0 and 1.
    """
    # Convert to boolean for logical operations
    mask1_bool = mask1 > 127
    mask2_bool = mask2 > 127

    intersection = np.logical_and(mask1_bool, mask2_bool).sum()
    union = np.logical_or(mask1_bool, mask2_bool).sum()

    iou = intersection / union if union != 0 else 0.0
    return iou

In [ ]:
import os
import json
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import cv2

# Setup paths
checkpoint_dir = "/kaggle/working"
os.makedirs(checkpoint_dir, exist_ok=True)

json_path = os.path.join(checkpoint_dir, "iou_accuracy_results.json")
csv_path = os.path.join(checkpoint_dir, "iou_accuracy_results.csv")

# --- Metric functions ---
def calculate_iou(mask1, mask2):
    mask1_bool = mask1 > 127
    mask2_bool = mask2 > 127
    intersection = np.logical_and(mask1_bool, mask2_bool).sum()
    union = np.logical_or(mask1_bool, mask2_bool).sum()
    return intersection / union if union != 0 else 0.0

def calculate_accuracy(mask1, mask2):
    return np.mean((mask1 > 127) == (mask2 > 127))

def calculate_precision_recall_f1(mask_pred, mask_true):
    mask_pred = mask_pred > 127
    mask_true = mask_true > 127
    tp = np.logical_and(mask_pred, mask_true).sum()
    fp = np.logical_and(mask_pred, ~mask_true).sum()
    fn = np.logical_and(~mask_pred, mask_true).sum()
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return precision, recall, f1

def combine_masks(predicted_masks, threshold):
    stack = np.stack(predicted_masks, axis=0)
    count_white = np.sum(stack, axis=0)
    return (count_white >= threshold).astype(np.uint8) * 255

# Resume from saved results
if os.path.exists(json_path):
    with open(json_path, "r") as f:
        saved_results = json.load(f)
else:
    saved_results = {}

saved_indices = set(map(int, saved_results.keys()))
ious, accuracies, f1s = [], [], []

# --- Main Loop (only indices 200 to 299) ---
for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
    if idx < 200 or idx >= 300:
        continue  # Process only range [200, 299]

    if idx in saved_indices:
        continue  # Skip already processed

    image_path = row["image_file_path"]
    mask_path = row["ROI_mask_file_path"]

    try:
        gt_mask = np.array(Image.open(mask_path).convert("L"))
        predicted_masks = []

        for n in range(2, 10):
            _, pred = predict_full_image_mask(model, image_path, n=n)
            bin_mask = (pred > 0.5).astype(np.uint8)
            predicted_masks.append(bin_mask)

        best_iou = 0
        best_acc = 0
        best_precision = 0
        best_recall = 0
        best_f1 = 0

        for t in range(2, 10):
            combined_mask = combine_masks(predicted_masks, threshold=t)
            resized = np.array(Image.fromarray(combined_mask).resize(gt_mask.shape[::-1], Image.NEAREST))

            # Largest component version
            num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(resized.astype(np.uint8), connectivity=8)
            if num_labels > 1:
                largest_label = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
                largest_component = (labels == largest_label).astype(np.uint8) * 255
            else:
                largest_component = resized

            for mask in [resized, largest_component]:
                iou = calculate_iou(mask, gt_mask)
                acc = calculate_accuracy(mask, gt_mask)
                precision, recall, f1 = calculate_precision_recall_f1(mask, gt_mask)

                if iou > best_iou:
                    best_iou = iou
                    best_acc = acc
                    best_precision = precision
                    best_recall = recall
                    best_f1 = f1

        saved_results[idx] = {
            "iou": best_iou,
            "accuracy": best_acc,
            "precision": best_precision,
            "recall": best_recall,
            "f1_score": best_f1
        }

        ious.append(best_iou)
        accuracies.append(best_acc)
        f1s.append(best_f1)

        # Save checkpoint
        with open(json_path + ".tmp", "w") as f:
            json.dump(saved_results, f)
        os.replace(json_path + ".tmp", json_path)

        pd.DataFrame.from_dict(saved_results, orient="index").to_csv(csv_path)

        if idx % 5 == 0:
            print(f"[Checkpoint] {idx} — Avg IoU: {np.mean(ious):.4f}, Acc: {np.mean(accuracies):.4f}, F1: {np.mean(f1s):.4f}")

    except Exception as e:
        print(f"Error at index {idx}: {e}")
        continue

# Final save
with open(json_path, "w") as f:
    json.dump(saved_results, f)

pd.DataFrame.from_dict(saved_results, orient="index").to_csv(csv_path)

# Final metrics display
df = pd.DataFrame.from_dict(saved_results, orient="index")
print("\n--- Final Evaluation Metrics ---")
print(f"Average IoU:       {df['iou'].mean():.4f}")
print(f"Average Accuracy:  {df['accuracy'].mean():.4f}")
print(f"Average Precision: {df['precision'].mean():.4f}")
print(f"Average Recall:    {df['recall'].mean():.4f}")
print(f"Average F1 Score:  {df['f1_score'].mean():.4f}")